# PricingModelSpec: C with shared transforms

C is the preferred design. Keep dataset details with the data, use a flat model
spec, and declare transformations once with `apply_transforms`. B is rejected.

| Decision | Current direction |
| :--- | :--- |
| DataFrame variable | `df` |
| Spec layout | Flat fields with short comments |
| Transformations | Declare once; reuse for data preparation and export |
| Offset | Fixed coefficient of 1, with the same transformation metadata as other inputs |
| Dataset provenance | Reuse the dataset saved during ingestion |

Start with the shared preparation and revised C below. Revised A remains an alternative. The original A, B, and C examples remain
later in the notebook for comparison. The original A is the runnable baseline.

**Implementation status:** the approved C design is now used by the package
scaffolder. `PricingDataset`, `apply_transforms`, `Log`, `Log1p`, `Clip`, and
the flat spec arguments are available. See the [generic training template](../../src/pricing_pipeline/resources/scaffold/notebooks/03_model_training.ipynb)
and [notebook guide](../../docs/notebooks/README.md). The examples below preserve
the design discussion; earlier alternatives remain sketches.

Executable cells create six synthetic rows, construct today's spec, and show
comparison tables. They do not train, connect to SQL, publish, or download data.
Select this repository's `.venv` kernel to run them.

## Shared preparation for revised A and C

Both options use exactly the same preparation and native SuperGLM estimator.
This example assumes `df` contains the source columns shown in the runnable
baseline below. For C, load the dataset before this preparation step.

### Define the transforms

Declare each transform once. Use it to prepare `df` and record it with the
workbook and SQL rating tables.

```python
transforms = {
    "log_density": Log1p("density"),
    "capped_vehicle_age": Clip("vehicle_age", upper=20),
    "log_exposure": Log("exposure"),
}

df = apply_transforms(df, transforms)

features = {
    "log_density": Spline(k=5),
    "capped_vehicle_age": Spline(k=5),
    "region": Categorical(),
}

estimator = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    discrete=True,
    n_bins=64,
    features=features,
)
```

The proposed helper would return a prepared DataFrame and retain source columns.
The explicit `transforms` variable carries the recipe between preparation and
the spec; it does not depend on hidden DataFrame attributes. Feature definitions
remain ordinary SuperGLM objects. Unchanged columns need no transformation entry.

`log_exposure` has the same transformation description as `log_density`.
The spec separately identifies it as an offset, whose coefficient is fixed at 1.
No special source or mandatory label field is needed for it. Display labels
could be optional for any feature.

### What is `Log1p`?

`Log1p("density")` means the natural logarithm of `1 + density`. The name
comes from NumPy's `np.log1p`. It gives 0 at density 0, whereas `log(0)` is
not finite. For example, density 9 becomes `log(10)`, about 2.303. NumPy also
calculates it accurately when the input is very close to zero.

`Log("exposure")` means `log(exposure)`, with no added 1. Positive exposure
is required. Changing this offset to `log1p(exposure)` would change the model.
`Log1p` and `Log` here are proposed helper names; NumPy's functions already exist.


## Revised A. Keep the data details in the flat spec

All configuration is visible in the training notebook. Use the shared
preparation above, then describe the model with short comments.

```python
spec_a = PricingModelSpec(
    # Model identity in SQL.
    name="MOTOR_FREQUENCY",
    label="Motor claim frequency",
    model_type="superglm_poisson",
    deployment_slot="MOTOR_UAT",

    # What to fit and how to validate it.
    target="claims",
    features=tuple(features),
    validation=ValidationSplitConfig.kfold(n_splits=5, random_state=42),

    # Where the data came from and how to identify each row.
    dataset_name="motor_policies",
    source_system="policy_warehouse",
    pk_columns=("policy_id",),
    data_as_of_column="data_as_of",

    # Preparation shared with the workbook and SQL package.
    transforms=transforms,

    # Fixed effect for exposure; weights for rating-table summaries.
    offset_column="log_exposure",
    export_weight_column="exposure",
)
```

This retains A's familiar structure while removing the separate offset source
and label arguments. It is longer than C's training spec, but you can read the
data identity without looking in another notebook or inspecting another object.

## Revised C. Reuse the dataset, preferred

Save the data details during ingestion. Reuse them when fitting a model.

### Record the dataset once during ingestion

```python
dataset = PricingDataset(
    df=df,
    name="motor_policies",
    source="policy_warehouse",
    key="policy_id",
    as_of="data_as_of",
)
dataset.save(MODEL_DIR / ".local" / "dataset.joblib")
```

### Load it during training

```python
dataset = PricingDataset.load(MODEL_DIR / ".local" / "dataset.joblib")
df = dataset.df
```

Run the shared preparation, `df = apply_transforms(df, transforms)`, then create
the flat spec. The dataset describes the ingested source; `df` is the prepared
data supplied to fitting. The implementation would need to verify their row
identity and transformation relationship before publication.

```python
spec_c = PricingModelSpec(
    # Model name and SQL destination.
    name="MOTOR_FREQUENCY",
    label="Motor claim frequency",
    model_type="superglm_poisson",
    deployment_slot="MOTOR_UAT",

    # Training data.
    dataset=dataset,

    # Target, features, and validation.
    target="claims",
    features=tuple(features),
    validation=ValidationSplitConfig.kfold(n_splits=5, random_state=42),

    # Save the transforms with the rating tables.
    transforms=transforms,

    # Add log exposure with coefficient 1.
    offset_column="log_exposure",

    # Weight the rating-table summaries by exposure.
    export_weight_column="exposure",
)
```

C no longer needs the original proposal's `prepared` object or `DerivedFeature`
wrappers. Its extra concept is the reusable dataset. It keeps the benefits of
A's comments and native SuperGLM feature definitions.

### Why `tuple(features)`?

`features` is a dictionary of column names and SuperGLM feature definitions.
`tuple(features)` takes its keys in the order they were written. The estimator
gets the full dictionary; the spec gets the column names.

A list preserves that order too. Today's constructor accepts a list at runtime
and stores a tuple, though its type annotation currently asks for a tuple.
The proposed API could accept lists explicitly. Avoid a set here: it does not
preserve insertion order and removes duplicates before validation can flag them.


## Why C

| Question | Revised A | Revised C |
| :--- | :--- | :--- |
| Where do I enter dataset provenance? | In the training spec | During ingestion |
| Can I read the full data identity in this spec? | Yes | Inspect the referenced dataset |
| What happens when several models use the same data? | Each spec supplies the data fields | Each spec references the same dataset |
| Does fitting use the shared transformation recipe? | Yes, proposed | Yes, proposed |
| Do I need nested model or feature wrappers? | No | No |

C is preferred because dataset details can be entered during ingestion and
reused across models. It keeps A's flat layout and simple comments. A remains
a useful comparison for examples that need all settings in one place.

### What export support would mean

Both proposals need the same implementation work. The workbook would describe
the source, transformation, and rating-table units. SQL would store the recipe
and either execute supported transformations or require prepared input columns.
Renaming a rating-table column alone would not convert its boundaries or values.
No automatic Python-to-SQL translation is claimed by these sketches.

## Original examples

The examples below preserve the first comparison. Original A runs with today's
API, so it still contains the offset source and label fields we propose removing
from analyst-facing configuration. Original B is rejected. Original C shows the
earlier, more object-heavy preparation design.

## The model used in every option

Predict claim count with a Poisson model and `log(exposure)` as the offset.
Use log density, vehicle age capped at 20, and region as model features.
Exposure also weights the rating-table summaries; it is not a fitting sample
weight in this count-and-offset formulation.

Every option keeps the same dataset identity, policy key, snapshot date,
model identity, and intended SQL deployment slot. Five-fold validation with
seed 42 is a proposed training choice. These six rows only illustrate preparation.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from superglm import Categorical, Spline, SuperGLM

from pricing_pipeline.models.config import ValidationSplitConfig
from pricing_pipeline.notebook import PricingModelSpec

df = pd.DataFrame({
    "policy_id": [1, 2, 3, 4, 5, 6],
    "claims": [0, 1, 0, 2, 0, 1],
    "exposure": [1.0, 0.5, 1.0, 0.75, 0.25, 1.0],
    "density": [0.0, 15.0, 100.0, 500.0, 1200.0, 3000.0],
    "vehicle_age": [0, 3, 8, 15, 25, 99],
    "region": ["A", "B", "A", "C", "B", "C"],
    "data_as_of": ["2026-09-01"] * 6,
})
display(df)

,policy_id,claims,exposure,density,vehicle_age,region,data_as_of
0,1,0,1.00,0.0,0,A,2026-09-01
1,2,1,0.50,15.0,3,B,2026-09-01
2,3,0,1.00,100.0,8,A,2026-09-01
3,4,2,0.75,500.0,15,C,2026-09-01
4,5,0,0.25,1200.0,25,B,2026-09-01
5,6,1,1.00,3000.0,99,C,2026-09-01


## A. One shorter spec

**Available now.** Keep ordinary pandas code, omit unused options, and define
the estimator features once. Comments divide the spec into decisions an analyst
can recognise. The model still has a single configuration object.

### Prepare the columns

Retain the source columns so original values remain available for review.

In [2]:
df = df.assign(
    log_density=np.log1p(df["density"]),
    capped_vehicle_age=df["vehicle_age"].clip(upper=20),
    log_exposure=np.log(df["exposure"]),
)

features = {
    "log_density": Spline(k=5),
    "capped_vehicle_age": Spline(k=5),
    "region": Categorical(),
}

display(df[["density", "log_density", "vehicle_age", "capped_vehicle_age",
            "exposure", "log_exposure"]])

,density,log_density,vehicle_age,capped_vehicle_age,exposure,log_exposure
0,0.0,0.000000,0,0,1.00,0.000000
1,15.0,2.772589,3,3,0.50,-0.693147
2,100.0,4.615121,8,8,1.00,0.000000
3,500.0,6.216606,15,15,0.75,-0.287682
4,1200.0,7.090910,25,20,0.25,-1.386294
5,3000.0,8.006701,99,20,1.00,0.000000


### Describe the model and its data

`pk_columns` identifies individual records. The dataset name, source, and
snapshot column describe which data was used. The offset fields connect the
fitting value to the exposure factor exported for rating.

In [3]:
spec_a = PricingModelSpec(
    # Model identity in SQL.
    name="MOTOR_FREQUENCY",
    label="Motor claim frequency",
    model_type="superglm_poisson",
    deployment_slot="MOTOR_UAT",

    # What to fit and how to validate it.
    target="claims",
    features=tuple(features),
    validation=ValidationSplitConfig.kfold(n_splits=5, random_state=42),

    # Where the data came from and how to identify each row.
    dataset_name="motor_policies",
    source_system="policy_warehouse",
    pk_columns=("policy_id",),
    data_as_of_column="data_as_of",

    # Exposure for fitting and rating-table summaries.
    offset_column="log_exposure",
    offset_source_column="exposure",
    offset_label="log(exposure)",
    export_weight_column="exposure",
)

estimator_a = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    discrete=True,
    n_bins=64,
    features=features,
)
display({"Model": spec_a.label, "Target": spec_a.target,
         "Features": list(spec_a.features), "Validation": "5 folds, seed 42"})

{'Model': 'Motor claim frequency',
 'Target': 'claims',
 'Features': ['log_density', 'capped_vehicle_age', 'region'],
 'Validation': '5 folds, seed 42'}

### What this improves, and what remains awkward

You can copy this into today's workflow. There are no unused `None` options,
and feature names come from the same dictionary passed to SuperGLM.

The spec still has many fields. More importantly, it records the offset's
source relationship but does not record the ordinary feature transformations.
Keeping `density` in `df` does not teach the exporter that `log_density` was
derived from it. That connection still lives in your pandas code.

**Review prompt:** Is this readable enough once the comments are present, or
does one long constructor still make it hard to find what you need?

## B. Group the spec by purpose, rejected

**Rejected design, retained for comparison.** Keep one top-level `PricingModelSpec`, but group related
fields into named sections. Each transformed feature declares its source and
operation beside its SuperGLM feature type. The declared operations would
prepare the model columns and supply transformation metadata to publication.

This option replaces A's manual transformation cell. It starts with the raw
columns in `df`; it does not transform an already transformed value again.

```python
spec_b = PricingModelSpec(
    name="MOTOR_FREQUENCY",
    label="Motor claim frequency",
    deployment_slot="MOTOR_UAT",

    data=DatasetSpec(
        name="motor_policies",
        source="policy_warehouse",
        key="policy_id",
        as_of="data_as_of",
    ),

    model=PoissonModel(
        target="claims",
        features={
            "log_density": Feature(
                source="density", transform=Log1p(), term=Spline(k=5),
            ),
            "capped_vehicle_age": Feature(
                source="vehicle_age", transform=Clip(upper=20), term=Spline(k=5),
            ),
            "region": Categorical(),
        },
        offset=LogOffset(source="exposure", output="log_exposure"),
        export_weight="exposure",
        selection_penalty=0.0,
        discrete=True,
        n_bins=64,
    ),

    validation=KFold(splits=5, seed=42),
)
```

`PoissonModel` would create the estimator and select the existing
`superglm_poisson` model-type identifier. `LogOffset` would generate the offset
column and its export description. Fitting would use `fit_reml`, matching A's
current default. These are explicit proposed conveniences, not inferred facts.

### What this improves, and what it costs

Related decisions sit together. You can see a transformed feature's source,
operation, and fitted term on the same lines. Unchanged features stay short.

There is more nesting and a new model wrapper to learn. It could force us to
keep adding wrapper arguments whenever analysts need another SuperGLM option.
The total amount of typing is not necessarily smaller than A.

**Review prompt:** Do the sections help you scan the code, or do they just add
more brackets and class names?

## C. Keep dataset details with the data

**Proposed API.** Record provenance during ingestion, then refer to it in the
training notebook. Keep native SuperGLM construction visible. A small helper
would handle only the features that need preparation.

### In the ingestion notebook

This is the full setup that makes the later model spec shorter. These fields
still have to be supplied once; they have not disappeared.

```python
dataset = PricingDataset(
    df=df,
    name="motor_policies",
    source="policy_warehouse",
    key="policy_id",
    as_of="data_as_of",
)
dataset.save(MODEL_DIR / ".local" / "dataset.joblib")
```

### In the training notebook

```python
dataset = PricingDataset.load(MODEL_DIR / ".local" / "dataset.joblib")
df = dataset.df

features = {
    "log_density": DerivedFeature(
        source="density", transform=Log1p(), term=Spline(k=5),
    ),
    "capped_vehicle_age": DerivedFeature(
        source="vehicle_age", transform=Clip(upper=20), term=Spline(k=5),
    ),
    "region": Categorical(),
}

prepared = prepare_features(
    df,
    features=features,
    offset=LogOffset(source="exposure", output="log_exposure"),
)
df = prepared.df

spec_c = PricingModelSpec(
    name="MOTOR_FREQUENCY",
    label="Motor claim frequency",
    model_type="superglm_poisson",
    deployment_slot="MOTOR_UAT",
    dataset=dataset,
    target="claims",
    inputs=prepared,
    export_weight="exposure",
    validation=KFold(splits=5, seed=42),
)

estimator_c = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    discrete=True,
    n_bins=64,
    features=prepared.model_features,
)
```

The proposed `prepared` result would hold the prepared `df`, native SuperGLM
feature definitions, and the source/transformation/offset metadata. The model
spec would use the metadata without asking you to type each feature name again.
Publication would verify that the prepared data and declarations still agree.

`PricingDataset` would preserve provenance across notebooks. Today's
`save_model_frame` / `load_model_frame` helpers do not provide this proposed
dataset object. None of these declarations connects to SQL by itself.

### What this improves, and what it costs

The training spec focuses on model choices. Native SuperGLM options remain
available. The dataset can be reused by multiple candidate models, and each
derived feature is declared once.

There are two extra objects, `dataset` and `prepared`, to understand. This
is shorter at the training-spec cell because some work moved to ingestion
and preparation. The complete workflow still has that work.

**Review prompt:** Does the shorter model cell justify these two objects, or
would you rather keep everything together as in A or B?

## The transformation information we need in SQL

This is a preview of the information B and C would record. It is not output
from an implemented transformation adapter. The calculations shown in A are real.

A renamed export column is insufficient. For example, raw density must go
through `log1p` before a rating table expressed in log-density units can use it.

In [4]:
display(pd.DataFrame([
    {"Role": "Feature", "Source": "density", "Model column": "log_density",
     "Preparation": "log1p(density)", "Term": "Spline(k=5)"},
    {"Role": "Feature", "Source": "vehicle_age", "Model column": "capped_vehicle_age",
     "Preparation": "min(vehicle_age, 20)", "Term": "Spline(k=5)"},
    {"Role": "Feature", "Source": "region", "Model column": "region",
     "Preparation": "Unchanged", "Term": "Categorical"},
    {"Role": "Offset", "Source": "exposure", "Model column": "log_exposure",
     "Preparation": "log(exposure)", "Term": "Fixed coefficient of 1"},
]))

,Role,Source,Model column,Preparation,Term
0,Feature,density,log_density,log1p(density),Spline(k=5)
1,Feature,vehicle_age,capped_vehicle_age,"min(vehicle_age, 20)",Spline(k=5)
2,Feature,region,region,Unchanged,Categorical
3,Offset,exposure,log_exposure,log(exposure),Fixed coefficient of 1


The SQL integration would need to reproduce these operations or explicitly
require prepared input columns. The helper names above do not imply that
Python-to-SQL translation already exists. The exposure factor must be applied
once during scoring.

For source-scale charts, we can transform an original-value grid before
evaluating an effect. That is a reporting operation. Converting a published
scoring table into original units is a separate change that needs validation.

### Where a small helper must stop

| Case | What the definition must preserve |
| :--- | :--- |
| Log density | Source, operation, valid domain, missing-value policy |
| Capped age | Source and cap, including behavior beyond the cap |
| `income / household_size` | Both source columns and zero-denominator behavior |
| Learned category encoding | Fitted state and training-fold isolation |
| Custom Python transformation | Explicit preparation requirement; no automatic SQL conversion |

The first prototype should cover fixed transformations such as log and clip.
It should not pretend that learned encodings can be fitted once before
cross-validation. Ordinary pandas preparation should remain available.

## Clearer wording around fitting and publishing

These are proposed notebook headings, using the current callable names.
They belong immediately above the relevant code cell in the real workflow.

### Fit, validate, and prepare a rating package

Fits validation models, then the final model on all training rows. Creates
the rating workbook and saved candidate, and records the dataset and split
evidence in the selected database. Review validation metrics before publishing.

```python
# Current API. Requires a database connection and registered model.
candidate = fit_model(
    pricing,
    model=model,
    frame=df,
    superglm_model=estimator_a,
)
display(candidate.metrics)
```

`frame=df` uses the existing keyword while keeping the notebook variable `df`.
`fit_candidate(..., df=df, estimator=...)` is a possible future spelling.
It would still include validation, export, and evidence writes, as described above.

### Save this model version to the selected database

Checks the package and saves its rating tables, model run, and validation
results. The destination is SQL Server in remote mode and SQLite in local
mode. An equivalent package may be reused. Activation happens later in the
deployment notebook.

```python
# Current API. Writes to the selected database.
published = save_model_version(pricing, candidate)
display({
    "Model": published.model_name,
    "Version": published.package_version,
    "Status": published.package_status,
    "Reused equivalent version": published.deduplicated,
})
```

The build and publication calls above are examples, not executable cells in
this comparison notebook. No database connection is created here.

## Original comparison

| Question | A | B | C |
| :--- | :--- | :--- | :--- |
| Works with today's public API? | Yes | No | No |
| Ordinary pandas preparation? | Yes | Replaced for declared transforms | Still available; helper is optional |
| Source mapping for derived features? | Only in pandas code | Beside each feature | Beside each feature |
| Where is dataset provenance? | In the model spec | Nested in the model spec | Recorded during ingestion |
| Where are SuperGLM options? | Native estimator | New model wrapper | Native estimator |
| Main cost | Long spec and unrecorded transformations | Nesting and wrapper maintenance | Dataset and preparation objects |

The revised examples at the top combine the shared preferences from A and C.
C is preferred. B is rejected.

An analyst should be able to complete the notebook using its examples and
explanations. A future agent instruction file should describe that same
workflow and its constraints.

## Notes for the next revision

Edit this cell as you compare the examples.

- The option I would start from:
- Names that still make me stop and think:
- Fields I want to see together:
- Information I would rather enter during ingestion:
- A real transformation this design needs to handle:
- My preferred names for fitting and publishing: